<a href="https://colab.research.google.com/github/loulakanani/machine-learning-in-finance/blob/main/5_Early_Default_Warning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Early Default Warning & Account Intervention: A Transparent Logistic Regression Baseline

## 1. Business Problem, Prediction Framework & Information Boundary

* **Business Problem & Decision:** A credit card issuer monitors active cardholders monthly to prevent near-term payment delinquency. The business decision is to flag high-risk accounts for **early preventive intervention** (e.g., proactive payment reminders, temporary credit-limit freezes, or restructuring offers) before default occurs, balancing operational outreach costs against expected credit losses.
* **Observation Unit ($i$):** One individual active credit card account ($N = 1,000$) observed at the monthly reporting snapshot.
* **Positive Event ($y_i = 1$):** Payment default in the following billing cycle (`default_next_month = 1`; $0$ indicates timely payment / no default).
* **Outcome Horizon ($t_1$):** One billing cycle forward (next month).
* **Information Cutoff ($t_0$):** All 18 predictor features (credit limit, current and lagged bill/payment amounts, utilization ratios, and 6-month delinquency indicators) are measured strictly at the reporting cutoff $t_0$, prior to observing next month's repayment outcome. The `ID` column is a row identifier and is excluded from modelling.


## 2. Data Loading, Audit & Quality Observations


In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
)

df = pd.read_csv("credit_ml_sample.csv")

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")

# Audit table: data types, missing values, duplicates, zeros, min/max
audit_df = pd.DataFrame({
    "Dtype": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Missing_Pct (%)": (df.isna().mean() * 100).round(2),
    "Zero_Count": (df == 0).sum(),
    "Min": df.min(numeric_only=True),
    "Max": df.max(numeric_only=True),
})
display(audit_df)

# Target prevalence
TARGET = "default_next_month"
target_summary = pd.DataFrame({
    "Client_Count": df[TARGET].value_counts(),
    "Prevalence (%)": (df[TARGET].value_counts(normalize=True) * 100).round(2),
})
target_summary.index = ["No Default (0)", "Default (1)"]
print("\nTarget Class Distribution:")
display(target_summary)


Dataset Shape: 1,000 rows x 20 columns



,Dtype,Missing_Count,Missing_Pct (%),Zero_Count,Min,Max
ID,int64,0,0.0,0,1.000000,1.000000e+03
LIMIT_BAL,int64,0,0.0,0,10000.000000,7.000000e+05
PAY_0,int64,0,0.0,473,-2.000000,8.000000e+00
BILL_AMT1,int64,0,0.0,74,-14386.000000,5.077260e+05
BILL_AMT2,int64,0,0.0,99,-13543.000000,5.092290e+05
PAY_AMT1,int64,0,0.0,183,0.000000,1.996460e+05
PAY_AMT2,int64,0,0.0,204,0.000000,2.851380e+05
default_next_month,int64,0,0.0,786,0.000000,1.000000e+00
utilization,float64,0,0.0,74,-0.071930,3.318933e+00
total_bill_6m,int64,0,0.0,35,-336259.000000,2.926653e+06



Target Class Distribution:


,Client_Count,Prevalence (%)
No Default (0),786,78.6
Default (1),214,21.4


### Data Audit & Quality Findings
* **Target Prevalence:** Out of $N = 1,000$ accounts, **21.40%** ($214$ clients) experienced default (`y = 1`), while **78.60%** ($786$ clients) did not (`y = 0`). Consequently, a naive majority-class baseline achieves $78.60\%$ accuracy while missing $100\%$ of defaults, confirming why probability calibration and cost-sensitive thresholds are required.
* **Data Quality Observation:** The engineered ratio `payment_to_bill_6m` contains **missing values (`NaN`)** whenever a client's 6-month total bill (`total_bill_6m`) equals zero (zero-division undefined ratio). Furthermore, several accounts exhibit negative bill amounts (`BILL_AMT1 < 0`) and negative utilization ratios, reflecting credit balances from overpayments, and features span vastly different scales (e.g., `LIMIT_BAL` up to $700,000$ vs. binary flags in $\{0,1\}$). All imputation and scaling must therefore be encapsulated inside a training-only `Pipeline`.


## 3. Mathematical Specification of Logistic Regression

1. **Linear Predictor (Log-Odds Score):**
   $$\eta_i = \beta_0 + \sum_{j=1}^{p} \beta_j x_{ij} = \beta_0 + x_i^\top \beta$$
   where $x_{ij}$ is the value of feature $j$ for account $i$, $\beta_0$ is the intercept, $\beta_j$ is the slope coefficient for feature $j$, and $\eta_i \in \mathbb{R}$ is the linear risk score.

2. **Sigmoid Link Function (Probability):**
   $$p_i = \Pr(Y_i = 1 \mid x_i) = \sigma(\eta_i) = \frac{1}{1 + e^{-\eta_i}} \in (0, 1)$$
   where $p_i$ is the predicted probability of default and $\sigma(\cdot)$ maps the unbounded score $\eta_i$ into the unit interval.

3. **Odds and Logit Transformation:**
   $$\text{odds}_i = \frac{p_i}{1 - p_i} = e^{\eta_i} \iff \text{logit}(p_i) = \log\left(\frac{p_i}{1 - p_i}\right) = \eta_i$$
   where $\text{odds}_i$ represents the ratio of default probability to non-default probability.

4. **Bernoulli Likelihood:**
   $$\mathcal{L}(\beta) = \prod_{i=1}^{n} p_i^{y_i} (1 - p_i)^{1 - y_i}$$
   where $y_i \in \{0, 1\}$ is the observed binary default label and $n$ is the number of training rows.

5. **Binary Log-Loss (Negative Log-Likelihood Objective):**
   $$J(\beta) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$
   Minimising $J(\beta)$ (plus an optional regularisation penalty) rewards well-calibrated probabilities and heavily penalises confident wrong predictions.


## 4. Validation Design: Comparing Ridge ($L_2$), LASSO ($L_1$), and Elastic Net
To prevent data leakage, we split the data into **60% Train**, **20% Validation**, and **20% untouched Test** sets using stratified sampling. Hyperparameters ($C = 1/\lambda$) are tuned via 5-fold cross-validation strictly inside the training set, and models are compared on the validation set using **Log-Loss**, **Brier Score**, and **ROC-AUC**. The test set is never used for model or threshold selection.


In [19]:
# Separate features and target (exclude ID)
X = df.drop(columns=["ID", TARGET])
y = df[TARGET]

# Strict 60 / 20 / 20 stratified split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print(f"Train size: {len(X_train)} | Validation size: {len(X_val)} | Test size: {len(X_test)}")

# Candidate regularisation families (C = 1 / lambda in scikit-learn)
C_grid = [0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

candidates = {
    "Ridge (L2)": LogisticRegression(
        penalty="l2", solver="saga", max_iter=5000, random_state=42
    ),
    "LASSO (L1)": LogisticRegression(
        penalty="l1", solver="saga", max_iter=5000, random_state=42
    ),
    "Elastic Net (l1_ratio=0.5)": LogisticRegression(
        penalty="elasticnet", l1_ratio=0.5, solver="saga", max_iter=5000,
        random_state=42
    ),
}

val_results = []
fitted_pipelines = {}

for name, estimator in candidates.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logit", estimator),
    ])

    # Tune C strictly inside X_train
    cv = GridSearchCV(
        pipe,
        param_grid={"logit__C": C_grid},
        scoring="neg_log_loss",
        cv=5,
        n_jobs=-1,
    )
    cv.fit(X_train, y_train)
    best_model = cv.best_estimator_
    fitted_pipelines[name] = best_model

    # Predict probabilities on validation set
    p_val = best_model.predict_proba(X_val)[:, 1]
    n_nonzero = np.sum(np.abs(best_model.named_steps["logit"].coef_[0]) > 1e-6)

    val_results.append({
        "Model": name,
        "Selected C": cv.best_params_["logit__C"],
        "Implied lambda (1/C)": round(1.0 / cv.best_params_["logit__C"], 4),
        "Non-zero Features": f"{n_nonzero}/{X.shape[1]}",
        "Val Log-Loss": round(log_loss(y_val, p_val), 4),
        "Val Brier Score": round(brier_score_loss(y_val, p_val), 4),
        "Val ROC-AUC": round(roc_auc_score(y_val, p_val), 4),
    })

# Add naive majority-class probability baseline (constant p = train default rate)
p_naive = np.full(len(y_val), y_train.mean())
val_results.append({
    "Model": "Naive Base-Rate Baseline",
    "Selected C": np.nan,
    "Implied lambda (1/C)": np.nan,
    "Non-zero Features": f"0/{X.shape[1]}",
    "Val Log-Loss": round(log_loss(y_val, p_naive), 4),
    "Val Brier Score": round(brier_score_loss(y_val, p_naive), 4),
    "Val ROC-AUC": 0.5000,
})

val_summary_df = pd.DataFrame(val_results)
display(val_summary_df)


Train size: 600 | Validation size: 200 | Test size: 200


,Model,Selected C,Implied lambda (1/C),Non-zero Features,Val Log-Loss,Val Brier Score,Val ROC-AUC
0,Ridge (L2),0.01,100.0,18/18,0.4554,0.1440,0.7313
1,LASSO (L1),0.05,20.0,3/18,0.4570,0.1450,0.7473
2,Elastic Net (l1_ratio=0.5),0.05,20.0,7/18,0.4497,0.1423,0.7480
3,Naive Base-Rate Baseline,NaN,NaN,0/18,0.5205,0.1688,0.5000


### Validation Comparison & Model Selection (Requirement 4)

* **Winning Model — Elastic Net (`l1_ratio = 0.5`, $C = 0.05$, $\lambda = 20.0$):** Across all three regularised candidates, **Elastic Net** achieves the strongest validation performance across every metric, recording the lowest **Log-Loss ($0.4497$)**, the lowest **Brier Score ($0.1423$)**, and the highest **ROC-AUC ($0.7480$)**.
* **Comparison Against Naive Baseline:** All three logistic models substantially outperform the naive base-rate baseline ($\text{Log-Loss} = 0.5205$, $\text{Brier Score} = 0.1688$, $\text{ROC-AUC} = 0.5000$), confirming that the account-level repayment and utilization features carry genuine predictive signal.
* **Sparsity vs. Shrinkage Trade-Off:**
  * **Ridge ($L_2$)** retains all **18/18** features with heavy uniform shrinkage ($\lambda = 100.0$), yielding solid calibration ($\text{Log-Loss} = 0.4554$) but slightly lower discrimination ($\text{ROC-AUC} = 0.7313$) due to collinear noise.
  * **LASSO ($L_1$)** enforces aggressive feature selection, retaining only **3/18** predictors ($\lambda = 20.0$), which improves ranking ($\text{ROC-AUC} = 0.7473$) but slightly sacrifices probability calibration ($\text{Log-Loss} = 0.4570$).
  * **Elastic Net** strikes the optimal balance by retaining the **7/18** strongest features while shrinking correlated repayment predictors smoothly, making it our selected model for downstream coefficient interpretation and threshold analysis.


## 5. Coefficient Interpretation in Odds Language


In [20]:
# Select the best-performing model on Validation Log-Loss
best_model_name = val_summary_df.iloc[:3].sort_values("Val Log-Loss").iloc[0]["Model"]
selected_pipe = fitted_pipelines[best_model_name]
print(f"Selected Model based on Validation Log-Loss: {best_model_name}\n")

logit_step = selected_pipe.named_steps["logit"]
coef_table = pd.DataFrame({
    "Feature": X.columns,
    "Beta (Standardised)": logit_step.coef_[0],
    "Odds_Multiplier_exp(Beta)": np.exp(logit_step.coef_[0]),
}).sort_values(by="Beta (Standardised)", key=abs, ascending=False)

display(coef_table.round(4))


Selected Model based on Validation Log-Loss: Elastic Net (l1_ratio=0.5)



,Feature,Beta (Standardised),Odds_Multiplier_exp(Beta)
12,recent_overdue,0.3118,1.3659
10,overdue_count_6m,0.1433,1.1540
11,max_delay_6m,0.1284,1.1370
4,PAY_AMT1,-0.0472,0.9539
1,PAY_0,0.0348,1.0354
13,high_utilization,-0.0260,0.9743
17,log_limit,-0.0098,0.9903
0,LIMIT_BAL,0.0000,1.0000
3,BILL_AMT2,0.0000,1.0000
2,BILL_AMT1,0.0000,1.0000


### Coefficient Interpretation in Odds Language (Requirement 5)

* **Dominant Risk Driver (`recent_overdue`):** The strongest predictor selected by Elastic Net is `recent_overdue` with a standardised coefficient of $\beta = 0.3118$ and an odds multiplier of $\exp(0.3118) = 1.3659$:
  > *"Holding all other features fixed, a one-standard-deviation increase in `recent_overdue` multiplies the odds of default by $\exp(\beta) = 1.3659$ (an increase of $36.59\%$)."*
* **Secondary Delinquency & Repayment Signals:** Consistent with credit risk economics, cumulative 6-month delinquency frequency (`overdue_count_6m`, $\exp(\beta) = 1.1540$) and maximum delay (`max_delay_6m`, $\exp(\beta) = 1.1370$) substantially increase default odds, whereas larger recent repayments (`PAY_AMT1`, $\exp(\beta) = 0.9539$) and higher credit limits (`log_limit`, $\exp(\beta) = 0.9903$) reduce default odds.
* **Sparsity Effect:** Because Elastic Net combines $L_1$ and $L_2$ penalties, it sets **11 redundant features** (such as raw `LIMIT_BAL` and raw bill amounts) exactly to $\beta_j = 0$ ($\exp(0) = 1.0000$), retaining only the **7 most informative signals**.
* **Nonlinearity Caveat (Odds vs. Probability):** It is crucial to note that $\beta_j$ does **not** represent a constant percentage-point change in default probability $p_i$. Because the sigmoid link function $\sigma(\eta_i)$ is S-shaped, a one-unit increase in $\eta_i$ produces a larger probability shift for borrowers near the centre of the distribution ($p_i \approx 0.50$) than for borrowers at the extreme tails ($p_i \approx 0$ or $p_i \approx 1$).


## 6. Policy Threshold Comparison ($c_1 = 0.20$ vs. $c_2 = 0.50$)
Without refitting the model coefficients $\beta$, we evaluate two operational decision thresholds on the validation probabilities $p_i = \Pr(Y_i = 1 \mid x_i)$.


In [21]:
p_val_selected = selected_pipe.predict_proba(X_val)[:, 1]

threshold_rows = []
for c in [0.20, 0.50]:
    y_hat = (p_val_selected >= c).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_hat).ravel()
    threshold_rows.append({
        "Threshold (c)": c,
        "Flagged Positive (%)": round(y_hat.mean() * 100, 1),
        "True Negatives (TN)": tn,
        "False Positives (FP)": fp,
        "False Negatives (FN)": fn,
        "True Positives (TP)": tp,
        "Precision": round(precision_score(y_val, y_hat, zero_division=0), 4),
        "Recall (Sensitivity)": round(recall_score(y_val, y_hat), 4),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

# Final one-time evaluation on the untouched Test set using chosen threshold c = 0.20
CHOSEN_C = 0.20
p_test_final = selected_pipe.predict_proba(X_test)[:, 1]
y_test_hat = (p_test_final >= CHOSEN_C).astype(int)
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_test_hat).ravel()

print("\n--- FINAL UNTOUCHED TEST SET REPORT ---")
print(f"Test Log-Loss:    {log_loss(y_test, p_test_final):.4f}")
print(f"Test Brier Score: {brier_score_loss(y_test, p_test_final):.4f}")
print(f"Test ROC-AUC:     {roc_auc_score(y_test, p_test_final):.4f}")
print(f"Test Confusion Matrix (at c={CHOSEN_C}): TN={tn_t}, FP={fp_t}, FN={fn_t}, TP={tp_t}")


,Threshold (c),Flagged Positive (%),True Negatives (TN),False Positives (FP),False Negatives (FN),True Positives (TP),Precision,Recall (Sensitivity)
0,0.2,36.0,116,41,12,31,0.4306,0.7209
1,0.5,1.5,157,0,40,3,1.0000,0.0698



--- FINAL UNTOUCHED TEST SET REPORT ---
Test Log-Loss:    0.4385
Test Brier Score: 0.1380
Test ROC-AUC:     0.7845
Test Confusion Matrix (at c=0.2): TN=117, FP=40, FN=10, TP=33


## 7. Threshold Policy Trade-Off, Final Test Evaluation & Governance Limitations

### Threshold Comparison: False Positives vs. False Negatives (Requirement 6)
* **Strict Default Rule ($c = 0.50$):** Using the conventional $0.50$ cutoff flags only **1.5%** of validation accounts ($3$ clients). While this avoids any false alarms ($\text{FP} = 0$, $\text{Precision} = 1.0000$), it fails to detect **40 out of 43 actual defaulters** ($\text{FN} = 40$, $\text{Recall} = 6.98\%$), exposing the issuer to severe unmitigated credit losses.
* **Preventive Intervention Rule ($c = 0.20$):** Lowering the decision threshold to $c = 0.20$—without refitting the model coefficients $\beta$—flags **36.0%** of accounts ($72$ clients) for early intervention. **False Negatives drop sharply from $40$ to $12$** (boosting **Recall from $6.98\%$ to $72.09\%$**), capturing $31$ of the $43$ defaulters prior to delinquency. The operational trade-off is an increase in **False Positives from $0$ to $41$** ($\text{Precision} = 43.06\%$), meaning $41$ creditworthy cardholders receive unnecessary preventive reminders or temporary limit freezes.
* **Policy Recommendation ($c = 0.20$):** Because the financial loss of an unflagged default (FN) far outweighs the low operational cost of a proactive reminder or temporary limit freeze (FP), **$c = 0.20$** is selected as the operational decision policy.

### Final Untouched Test Set Evaluation
* Evaluating the frozen Elastic Net pipeline once on the held-out test set ($N_{\text{test}} = 200$) confirms strong generalisation: **Test Log-Loss = $0.4385$**, **Test Brier Score = $0.1380$**, and **Test ROC-AUC = $0.7845$**. At the chosen policy threshold $c = 0.20$, the model captures **33 out of 43 test defaulters** ($\text{TP} = 33$, $\text{FN} = 10$, $\text{Recall} = 76.74\%$) with $\text{FP} = 40$ and $\text{TN} = 117$.

### Leakage Risk & Sample Limitation (Requirement 7)
* **Leakage Risk:** Computing median imputation (`SimpleImputer`) or feature standardisation (`StandardScaler`) on the full $1,000$-row dataset prior to the train/validation/test split—or tuning $C$ and threshold $c$ using test labels—would leak out-of-sample distributional information into model estimation. Encapsulating all preprocessing inside a `scikit-learn` `Pipeline` fitted strictly on `X_train` and touching `X_test` only once eliminates this risk.
* **Sample Limitation:** The dataset comprises a relatively small cross-sectional sample ($N = 1,000$) from a single reporting period without macroeconomic indicators (e.g., unemployment or interest rate shifts) or recovery rates (Loss Given Default). Consequently, estimated coefficients capture conditional associations within this cohort and require ongoing calibration monitoring against concept drift before production deployment.

##AI Usage Disclosure
In accordance with academic integrity guidelines, an AI language assistant was used solely as a supplementary editorial and coding support tool. Specifically, it assisted with structuring the Markdown documentation, verifying scikit-learn pipeline syntax, and refining the clarity of the English text. All problem framing, data auditing, model selection decisions, economic interpretations, and final evaluations were conducted and verified independently by the author.